In [1]:
'''
Evaluate the performance of search algorithms 
Collect scores across all prompts and trials, and compute the overall statistics.
'''

import os, psutil, gc
import time 
import json
import pprint
import signal

from collections import defaultdict
import random

import numpy as np
np.set_printoptions(precision=4)
from scipy.stats import ttest_rel

In [2]:
from sal.config import Config

from datasets import load_dataset

# from core import best_of_n
from utils.load_data import load_data_prm800k
from utils import grader 
from utils import grader2
from utils import parser

In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_tokenizer_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_tokenizer_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"

In [4]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)


# ds_completions = load_completions(completions_dir)

# load random_seeds     
# random_seeds = np.loadtxt("random_seeds.txt").astype("int64")
# random_seeds = [int(seed) for seed in random_seeds]

1: 43
2: 90
3: 105
4: 128
5: 134


In [5]:
level = 4
num_questions = len(data_by_levels[level])  # level 4 has 128 questions

In [6]:
level = 4
num_trials = 5
metric_name = "pass1b"

print(f"\n-> cnt-mcts")
cnt_config_name = f"mcts--level-4--e31--n-4--d-20--nb-4--cpuct-2"
cnt_dir = f"results/mcts--level-{level}/{cnt_config_name}"
cnt_res = np.loadtxt(f"{cnt_dir}/{metric_name}_{cnt_config_name}.txt")
cnt_scores = []
for tidx in range(num_trials):
    cnt_scores.append(cnt_res[tidx*num_questions:(tidx+1)*num_questions])
cnt_scores = np.stack(cnt_scores, axis=1)
cnt_means = np.mean(cnt_scores, axis=1)
print(cnt_means)

print(f"\n-> embeds-mcts")
embeds_config_name = f"mcts--level-4--e71--n-4--d-20--nb-4--lam-0.01--dalpha-100.0--dbeta-1.0--ppl-True--normalize-True"
embeds_dir = f"results/mcts--level-{level}/{embeds_config_name}"
embeds_res = np.loadtxt(f"{embeds_dir}/{metric_name}_{embeds_config_name}.txt")
embeds_scores = []
for tidx in range(num_trials):
    embeds_scores.append(embeds_res[tidx*num_questions:(tidx+1)*num_questions])
embeds_scores = np.stack(embeds_scores, axis=1)
embeds_means = np.mean(embeds_scores, axis=1)
print(embeds_means)


for idx in range(num_questions):
    cnt_score = cnt_means[idx]
    embeds_score = embeds_means[idx]
    if embeds_score == 0 and cnt_score > 0:
        print(f"idx = {idx}: {embeds_score} {cnt_score}")


-> cnt-mcts
[0.2 0.8 1.  1.  0.2 1.  0.6 1.  1.  0.  1.  0.  0.6 0.  1.  1.  0.  0.8
 1.  0.2 1.  1.  0.6 0.  1.  0.  1.  1.  0.2 0.6 1.  0.6 1.  0.2 0.8 0.6
 1.  1.  1.  1.  0.8 0.8 0.6 0.8 1.  0.4 0.2 0.2 1.  0.8 0.  0.2 0.  0.
 0.2 1.  0.  1.  1.  0.  0.2 0.  0.2 0.2 1.  0.  1.  0.4 0.  1.  1.  0.8
 1.  0.6 0.  0.6 0.  0.  0.8 0.4 0.6 1.  0.4 1.  1.  1.  0.2 1.  1.  0.
 1.  1.  1.  1.  0.  1.  1.  0.  0.  0.  1.  1.  1.  0.8 0.6 1.  0.4 0.6
 1.  1.  0.6 0.8 1.  0.  0.2 0.  0.6 1.  1.  1.  0.2 1.  0.8 1.  1.  1.
 0.4 1. ]

-> embeds-mcts
[0.2 0.4 1.  1.  0.2 1.  0.6 1.  1.  0.  1.  0.2 0.8 0.  1.  1.  0.4 1.
 1.  0.  1.  1.  0.  0.  1.  0.  1.  1.  0.  0.6 1.  0.6 1.  0.2 0.8 0.6
 1.  1.  1.  1.  1.  0.8 0.8 1.  1.  0.4 0.  0.8 1.  1.  0.  0.4 0.  0.
 0.4 1.  0.  1.  1.  0.2 0.  0.  0.  0.4 1.  0.  1.  0.8 0.  1.  1.  0.8
 1.  0.8 0.  0.4 0.  0.  1.  0.  0.8 1.  0.4 1.  1.  0.8 0.2 1.  0.6 0.4
 1.  1.  1.  1.  0.  1.  1.  0.  0.  0.  0.8 1.  1.  0.6 0.4 1.  1.  0.4
 1.  1.  0.6 0.4 

In [7]:
stop

NameError: name 'stop' is not defined

In [ ]:

num_trials = 5
# num_questions = 128
metric_name = "pass1b"

config_names_arr = [
    "mcts--level-4--e31--n-4--d-20--nb-4--cpuct-2",
    "mcts--level-4--e71--n-4--d-20--nb-4--lam-0.01--dalpha-100.0--dbeta-1.0--ppl-True--normalize-True",
    # "mcts--v51--n-8--d-40--nb-1--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
]

nlen = len(config_names_arr)


results = []
for cidx in range(nlen):
    config_name = config_names_arr[cidx]
    print(f"\n")
    print(config_name)
    result_dir = f"results/mcts--level-{level}/{config_name}"
    try:
        
        res = np.loadtxt(f"{result_dir}/{metric_name}_{config_name}.txt")
        config_scores = []
        for tidx in range(num_trials):
            config_scores.append(res[tidx*num_questions:(tidx+1)*num_questions])
        print(len(config_scores))
        config_scores = np.stack(config_scores, axis=1)
        config_means = np.mean(config_scores, axis=1)
        config_easy_idxes = np.where(config_means == 1.0)[0]
        config_hard_idxes = np.where(config_means == 0.0)[0]
        print(f"easy ({len(config_easy_idxes)}): {config_easy_idxes}")
        print(f"hard ({len(config_hard_idxes)}): {config_hard_idxes}")

        res_mean = np.mean(res)
        res_std = np.std(res, ddof=1)/np.sqrt(num_trials*128)
        print(f"mean = {res_mean:0.4f} (\u00B1{res_std:0.4f})")
        
        
    except Exception as e:
        
        print(f"An unexpected error occurred: {e}")
        res = None
        results.append(res)
    

In [ ]:
num_trials = 2
num_questions = 128
metric_name = "pass1b"

bob_config = "bob--v11--n-40--d-40--level-4"
bob_res = np.loadtxt(f"results/{bob_config}/{metric_name}_{bob_config}.txt")
# results.append(res[:num_trials*128])
bob_scores = []
for tidx in range(num_trials):
    bob_scores.append(bob_res[tidx*num_questions:(tidx+1)*num_questions])
bob_scores = np.stack(bob_scores, axis=1)
bob_means = np.mean(bob_scores, axis=1)
bob_easy_idxes = set(np.where(bob_means == 1.0)[0])
# bob_hard_idxes = set(np.where(bob_means == 0.0)[0])
print(f"\n")
print(bob_config)
print(len(bob_easy_idxes), bob_easy_idxes)
# print(len(bob_hard_idxes), bob_hard_idxes)

mcts_config = "mcts--v12--n-8--d-40--nb-5--cpuct-2--level-4"
mcts_res = np.loadtxt(f"results/{mcts_config}/{metric_name}_{mcts_config}.txt")
mcts_scores = []
for tidx in range(num_trials):
    mcts_scores.append(mcts_res[tidx*num_questions:(tidx+1)*num_questions])
mcts_scores = np.stack(mcts_scores, axis=1)
mcts_means = np.mean(mcts_scores, axis=1)
mcts_easy_idxes = set(np.where(mcts_means == 1.0)[0])
# mcts_hard_idxes = np.where(mcts_means == 0.0)[0]

print(f"\n")
print(mcts_config)
print(len(mcts_easy_idxes), mcts_easy_idxes)

# mcts_easy_idxes = mcts_easy_idxes - bob_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)
# print(len(mcts_hard_idxes), mcts_hard_idxes)

mctsdiv_config = "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4"
mctsdiv_res = np.loadtxt(f"results/{mctsdiv_config}/{metric_name}_{mctsdiv_config}.txt")
mctsdiv_scores = []
for tidx in range(num_trials):
    mctsdiv_scores.append(mctsdiv_res[tidx*num_questions:(tidx+1)*num_questions])
mctsdiv_scores = np.stack(mctsdiv_scores, axis=1)
mctsdiv_means = np.mean(mctsdiv_scores, axis=1)
mctsdiv_easy_idxes = set(np.where(mctsdiv_means == 1.0)[0])
# mctsdiv_hard_idxes = set(np.where(mctsdiv_means == 0.0)[0])

print(f"\n")
print(mctsdiv_config)
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

# print(f"\n")
# print(f"common_easy_idxes")
# common_easy_idxes = bob_easy_idxes & mcts_easy_idxes & mctsdiv_easy_idxes
# print(len(common_easy_idxes), common_easy_idxes)

# print(f"\n")
# print(mcts_config)
# mcts_easy_idxes = mcts_easy_idxes - common_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)

print(f"\n")
print(mctsdiv_config)
# mctsdiv_easy_idxes = mctsdiv_easy_idxes - common_easy_idxes
# print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

mctsdiv_easy_idxes = mctsdiv_easy_idxes - mcts_easy_idxes
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)



In [ ]:
num_trials = 2
num_questions = 128
metric_name = "pass1b"

bob_config = "bob--v11--n-40--d-40--level-4"
bob_res = np.loadtxt(f"results/{bob_config}/{metric_name}_{bob_config}.txt")
# results.append(res[:num_trials*128])
bob_scores = []
for tidx in range(num_trials):
    bob_scores.append(bob_res[tidx*num_questions:(tidx+1)*num_questions])
bob_scores = np.stack(bob_scores, axis=1)
bob_means = np.mean(bob_scores, axis=1)
bob_easy_idxes = set(np.where(bob_means == 1.0)[0])
# bob_hard_idxes = set(np.where(bob_means == 0.0)[0])
print(f"\n")
print(bob_config)
print(len(bob_easy_idxes), bob_easy_idxes)
# print(len(bob_hard_idxes), bob_hard_idxes)

mcts_config = "mcts--v12--n-8--d-40--cpuct-2--level-4"
mcts_res = np.loadtxt(f"results/{mcts_config}/{metric_name}_{mcts_config}.txt")
mcts_scores = []
for tidx in range(num_trials):
    mcts_scores.append(mcts_res[tidx*num_questions:(tidx+1)*num_questions])
mcts_scores = np.stack(mcts_scores, axis=1)
mcts_means = np.mean(mcts_scores, axis=1)
mcts_easy_idxes = set(np.where(mcts_means == 1.0)[0])
# mcts_hard_idxes = np.where(mcts_means == 0.0)[0]

print(f"\n")
print(mcts_config)
print(len(mcts_easy_idxes), mcts_easy_idxes)

# mcts_easy_idxes = mcts_easy_idxes - bob_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)
# print(len(mcts_hard_idxes), mcts_hard_idxes)

mctsdiv_config = "mcts--v51--n-8--d-40--nb-1--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4"
mctsdiv_res = np.loadtxt(f"results/{mctsdiv_config}/{metric_name}_{mctsdiv_config}.txt")
mctsdiv_scores = []
for tidx in range(num_trials):
    mctsdiv_scores.append(mctsdiv_res[tidx*num_questions:(tidx+1)*num_questions])
mctsdiv_scores = np.stack(mctsdiv_scores, axis=1)
mctsdiv_means = np.mean(mctsdiv_scores, axis=1)
mctsdiv_easy_idxes = set(np.where(mctsdiv_means == 1.0)[0])
# mctsdiv_hard_idxes = set(np.where(mctsdiv_means == 0.0)[0])

print(f"\n")
print(mctsdiv_config)
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

# print(f"\n")
# print(f"common_easy_idxes")
# common_easy_idxes = bob_easy_idxes & mcts_easy_idxes & mctsdiv_easy_idxes
# print(len(common_easy_idxes), common_easy_idxes)

# print(f"\n")
# print(mcts_config)
# mcts_easy_idxes = mcts_easy_idxes - common_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)

print(f"\n")
print(mctsdiv_config)
# mctsdiv_easy_idxes = mctsdiv_easy_idxes - common_easy_idxes
# print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

mctsdiv_easy_idxes = mctsdiv_easy_idxes - mcts_easy_idxes
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)



In [ ]:
{96, 40, 42, 43, 81, 85, 24, 90, 127, 63}
{96, 5, 38, 42, 78, 84, 85, 24, 93, 127}

In [ ]:
{97, 34, 38, 71, 75, 114, 116, 63}
{96, 38, 103, 42, 78, 112, 82, 84, 85, 86, 24, 127}